# 4. Preparación y Transformación – AWS Glue / PySpark

En la pauta original, esta etapa corresponde a conectar Dataprep a BigQuery para realizar limpieza, transformación, validaciones y generación de atributos.

En esta implementación equivalente en AWS, la tabla RAW creada en Athena / Glue Data Catalog funciona como capa intermedia sobre los datos almacenados en Amazon S3. A partir de esa capa RAW, se utiliza PySpark como motor de transformación batch para preparar los datos y generar una salida procesada.

El notebook local se utiliza como herramienta de ejecución, documentación y evidencia del proceso, pero la fuente y el destino del pipeline se mantienen en AWS.

Flujo equivalente:

```text
Amazon S3 RAW
        ↓
Athena / Glue Data Catalog - Tabla RAW
        ↓
PySpark - Limpieza, validación y generación de atributos
        ↓
Amazon S3 processed
        ↓
Athena / Glue Data Catalog - Tabla procesada
```

Esta etapa cumple el rol equivalente a Dataprep, ya que toma los datos RAW, aplica reglas de preparación y genera una versión procesada lista para la capa analítica.

## 4.1 Reconexión al entorno AWS

Como este notebook continúa el trabajo realizado en etapas anteriores, no es necesario volver a crear el bucket, la tabla RAW ni la base de datos en Athena.

Sin embargo, al trabajar con un laboratorio AWS con credenciales temporales, sí es necesario volver a configurar la conexión local.

Para continuar desde este punto se deben realizar las siguientes acciones:

1. Actualizar el archivo `.env` con las nuevas credenciales temporales del laboratorio.
2. Cargar nuevamente las variables de entorno desde el notebook.
3. Crear nuevamente los clientes de AWS necesarios para conectarse a S3, Athena y STS.
4. Validar que las credenciales estén activas.
5. Confirmar que la tabla RAW creada anteriormente sigue disponible.

Esta configuración no replica el procesamiento anterior ni recrea recursos. Solo permite que el nuevo notebook pueda conectarse a los recursos existentes.

## 4.2 Variables de configuración utilizadas

El notebook utiliza un archivo `.env` para no escribir credenciales directamente dentro del código.

El archivo `.env` debe contener las credenciales temporales del laboratorio AWS y las rutas principales del proyecto:

```env
AWS_ACCESS_KEY_ID=your_access_key_here
AWS_SECRET_ACCESS_KEY=your_secret_key_here
AWS_SESSION_TOKEN=your_session_token_here
AWS_DEFAULT_REGION=us-east-1

S3_BUCKET=your_bucket_name_here
S3_RAW_PREFIX=raw/productive_data/
S3_RAW_KEY=raw/productive_data/productive_data.csv

ATHENA_DATABASE=salmonicultura_raw_db
ATHENA_RAW_TABLE=productive_data_raw
ATHENA_OUTPUT=s3://your_bucket_name_here/exports/athena_results/
CSV_DELIMITER=;

S3_PROCESSED_PREFIX=processed/productive_kpi/
```

El archivo `.env` no debe ser compartido ni subido a repositorios, ya que contiene credenciales sensibles.

Para la entrega, se puede incluir un archivo `.env.example` con la misma estructura, pero sin credenciales reales.

In [1]:
# Esta celda recarga las variables del archivo .env y reconstruye la configuración local.
# Es necesario ejecutarla al retomar el notebook porque las variables de Python se pierden al reiniciar el kernel.

from dotenv import load_dotenv
import os
import boto3
import time

load_dotenv(override=True)

AWS_ACCESS_KEY_ID = os.getenv("AWS_ACCESS_KEY_ID")
AWS_SECRET_ACCESS_KEY = os.getenv("AWS_SECRET_ACCESS_KEY")
AWS_SESSION_TOKEN = os.getenv("AWS_SESSION_TOKEN")
AWS_DEFAULT_REGION = os.getenv("AWS_DEFAULT_REGION")

S3_BUCKET = os.getenv("S3_BUCKET")
S3_RAW_PREFIX = os.getenv("S3_RAW_PREFIX")
S3_RAW_KEY = os.getenv("S3_RAW_KEY")

ATHENA_DATABASE = os.getenv("ATHENA_DATABASE")
ATHENA_RAW_TABLE = os.getenv("ATHENA_RAW_TABLE")
ATHENA_OUTPUT = os.getenv("ATHENA_OUTPUT")
CSV_DELIMITER = os.getenv("CSV_DELIMITER", ";")

print("Región AWS:", AWS_DEFAULT_REGION)
print("Bucket:", S3_BUCKET)
print("Tabla RAW:", ATHENA_RAW_TABLE)

Región AWS: us-east-1
Bucket: bigdata-salmonicultura-fabian
Tabla RAW: productive_data_raw


## 4.3 Carga de variables y creación de clientes AWS

En esta sección se cargan las variables del archivo `.env` y se crean los clientes necesarios para trabajar con AWS desde el notebook.

Los clientes utilizados son:

- `s3`: permite revisar archivos y guardar resultados en Amazon S3.
- `athena`: permite ejecutar consultas SQL sobre la tabla RAW.
- `sts`: permite validar que las credenciales temporales del laboratorio estén activas.

Esta configuración es necesaria al iniciar un nuevo notebook o al reiniciar el kernel.

In [2]:
# Esta celda vuelve a crear los clientes de AWS necesarios para continuar trabajando.
# No recrea recursos en AWS; solo permite que el notebook se conecte nuevamente a S3, Athena y STS.

s3 = boto3.client(
    "s3",
    aws_access_key_id=AWS_ACCESS_KEY_ID,
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
    aws_session_token=AWS_SESSION_TOKEN,
    region_name=AWS_DEFAULT_REGION
)

athena = boto3.client(
    "athena",
    aws_access_key_id=AWS_ACCESS_KEY_ID,
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
    aws_session_token=AWS_SESSION_TOKEN,
    region_name=AWS_DEFAULT_REGION
)

sts = boto3.client(
    "sts",
    aws_access_key_id=AWS_ACCESS_KEY_ID,
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
    aws_session_token=AWS_SESSION_TOKEN,
    region_name=AWS_DEFAULT_REGION
)

print("Clientes AWS creados correctamente.")

Clientes AWS creados correctamente.


## 4.4 Validación de credenciales AWS

Antes de continuar con la transformación de datos, se valida que las credenciales temporales del laboratorio AWS estén funcionando correctamente.

Para esto se utiliza el servicio STS mediante el método `get_caller_identity()`.

Si la validación responde con la cuenta y el ARN del rol activo, significa que el notebook está correctamente autenticado contra AWS.

In [3]:
# Esta celda valida que las nuevas credenciales temporales del laboratorio AWS estén funcionando.

identity = sts.get_caller_identity()

print("Credenciales AWS válidas.")
print("Cuenta AWS:", identity["Account"])
print("ARN:", identity["Arn"])

Credenciales AWS válidas.
Cuenta AWS: 601270941236
ARN: arn:aws:sts::601270941236:assumed-role/voclabs/user3277614=fabi.lecaros@duocuc.cl


## 4.5 Validación de disponibilidad de la tabla RAW

Luego de validar las credenciales, se comprueba que la tabla RAW creada en la etapa anterior sigue disponible en Athena.

Esta validación es importante porque permite confirmar que el notebook puede acceder a la fuente de datos que será utilizada para la transformación.

La comprobación se realiza mediante una consulta `COUNT(*)` sobre la tabla RAW:

```sql
SELECT COUNT(*) AS total_registros
FROM salmonicultura_raw_db.productive_data_raw;
```

Si la consulta devuelve el total de registros, se confirma que la tabla RAW está disponible y que se puede continuar con la preparación y transformación de datos.

In [4]:
# Esta función permite ejecutar consultas SQL en Athena desde el notebook.
# Se vuelve a definir porque las funciones también se pierden cuando se reinicia el kernel.

def run_athena_query(query, database=None):
    response = athena.start_query_execution(
        QueryString=query,
        QueryExecutionContext={
            "Database": database or ATHENA_DATABASE
        },
        ResultConfiguration={
            "OutputLocation": ATHENA_OUTPUT
        }
    )

    query_execution_id = response["QueryExecutionId"]

    while True:
        result = athena.get_query_execution(
            QueryExecutionId=query_execution_id
        )

        state = result["QueryExecution"]["Status"]["State"]

        if state in ["SUCCEEDED", "FAILED", "CANCELLED"]:
            break

        time.sleep(2)

    if state != "SUCCEEDED":
        reason = result["QueryExecution"]["Status"].get("StateChangeReason", "Sin detalle")
        raise Exception(f"La consulta falló. Estado: {state}. Motivo: {reason}")

    return query_execution_id

In [5]:
# Esta celda valida que la tabla RAW creada anteriormente sigue disponible en Athena.

count_query = f"""
SELECT COUNT(*) AS total_registros
FROM {ATHENA_DATABASE}.{ATHENA_RAW_TABLE}
"""

query_id = run_athena_query(count_query)

results = athena.get_query_results(QueryExecutionId=query_id)

for row in results["ResultSet"]["Rows"]:
    values = [col.get("VarCharValue", "") for col in row["Data"]]
    print(values)

['total_registros']
['783244']


## 4.6 Transformación del dataset RAW con PySpark desde Amazon S3

En esta sección se ejecuta la transformación batch utilizando PySpark como equivalente funcional a Dataprep.

La tabla RAW ya fue validada previamente en Athena / Glue Data Catalog. Para esta etapa, PySpark leerá directamente el archivo RAW desde Amazon S3 utilizando la ruta `s3a://`, aplicará reglas de limpieza, validación y generación de atributos, y guardará el resultado procesado nuevamente en Amazon S3.

De esta forma, el notebook actúa como herramienta de ejecución y evidencia, pero la fuente y el destino del procesamiento permanecen dentro de AWS.

Flujo de esta etapa:

```text
Amazon S3 RAW
        ↓
PySpark
        ↓
Limpieza y transformación
        ↓
Generación de indicadores
        ↓
Amazon S3 processed
```

In [6]:
# Esta celda carga las credenciales temporales del laboratorio AWS y las rutas del proyecto.
# Estas variables serán utilizadas por Spark para leer y escribir directamente en Amazon S3.

from dotenv import load_dotenv
import os

load_dotenv(override=True)

AWS_ACCESS_KEY_ID = os.getenv("AWS_ACCESS_KEY_ID")
AWS_SECRET_ACCESS_KEY = os.getenv("AWS_SECRET_ACCESS_KEY")
AWS_SESSION_TOKEN = os.getenv("AWS_SESSION_TOKEN")
AWS_DEFAULT_REGION = os.getenv("AWS_DEFAULT_REGION")

S3_BUCKET = os.getenv("S3_BUCKET")
S3_RAW_PREFIX = os.getenv("S3_RAW_PREFIX")
S3_RAW_KEY = os.getenv("S3_RAW_KEY")
S3_PROCESSED_PREFIX = os.getenv("S3_PROCESSED_PREFIX", "processed/productive_kpi/")

CSV_DELIMITER = os.getenv("CSV_DELIMITER", ";")

S3_RAW_PATH = f"s3a://{S3_BUCKET}/{S3_RAW_KEY}"
S3_PROCESSED_PATH = f"s3a://{S3_BUCKET}/{S3_PROCESSED_PREFIX}"

print("Ruta RAW en S3:", S3_RAW_PATH)
print("Ruta processed en S3:", S3_PROCESSED_PATH)
print("Delimitador CSV:", CSV_DELIMITER)

Ruta RAW en S3: s3a://bigdata-salmonicultura-fabian/raw/productive_data/productive_data.csv
Ruta processed en S3: s3a://bigdata-salmonicultura-fabian/processed/productive_kpi/
Delimitador CSV: ;


In [7]:
# Esta celda configura Java y Hadoop local desde el archivo .env.
# En Windows, PySpark necesita JAVA_HOME y HADOOP_HOME antes de iniciar la sesión Spark.

from dotenv import load_dotenv
import os

load_dotenv(override=True)

JAVA_HOME = os.getenv("JAVA_HOME")
HADOOP_HOME = os.getenv("HADOOP_HOME")

if JAVA_HOME:
    os.environ["JAVA_HOME"] = JAVA_HOME
    os.environ["PATH"] = os.path.join(JAVA_HOME, "bin") + os.pathsep + os.environ["PATH"]
    print("JAVA_HOME configurado:", os.environ["JAVA_HOME"])
else:
    print("JAVA_HOME no está definido en .env")

if HADOOP_HOME:
    os.environ["HADOOP_HOME"] = HADOOP_HOME
    os.environ["hadoop.home.dir"] = HADOOP_HOME
    os.environ["PATH"] = os.path.join(HADOOP_HOME, "bin") + os.pathsep + os.environ["PATH"]
    print("HADOOP_HOME configurado:", os.environ["HADOOP_HOME"])
else:
    print("HADOOP_HOME no está definido en .env")

JAVA_HOME configurado: C:\Program Files\Eclipse Adoptium\jdk-17.0.19.10-hotspot
HADOOP_HOME configurado: C:\hadoop


In [8]:
# Esta celda revisa la versión actual de PySpark instalada en el entorno.
# Si aparece una versión 4.x, conviene bajar a PySpark 3.5.1 para mayor compatibilidad con hadoop-aws 3.3.4.

import pyspark

print("Versión de PySpark:", pyspark.__version__)

Versión de PySpark: 3.5.1


In [9]:
# Esta celda inicia una sesión local de Spark configurada para conectarse a Amazon S3 mediante S3A.
# Se cargan las librerías necesarias de Hadoop AWS y se configuran las credenciales temporales del laboratorio.

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("PipelineBatchSalmoniculturaAWS") \
    .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.262") \
    .config("spark.hadoop.fs.s3a.access.key", AWS_ACCESS_KEY_ID) \
    .config("spark.hadoop.fs.s3a.secret.key", AWS_SECRET_ACCESS_KEY) \
    .config("spark.hadoop.fs.s3a.session.token", AWS_SESSION_TOKEN) \
    .config("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.TemporaryAWSCredentialsProvider") \
    .config("spark.hadoop.fs.s3a.endpoint", f"s3.{AWS_DEFAULT_REGION}.amazonaws.com") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .getOrCreate()

print("Sesión Spark iniciada correctamente.")

Sesión Spark iniciada correctamente.


In [10]:
# Esta celda lee directamente el archivo CSV RAW desde Amazon S3 usando PySpark.
# No se descarga manualmente el archivo: Spark accede al origen cloud mediante la ruta s3a://.

df_raw = spark.read \
    .option("header", True) \
    .option("sep", CSV_DELIMITER) \
    .option("encoding", "UTF-8") \
    .csv(S3_RAW_PATH)

print("Dataset RAW cargado desde S3 con PySpark.")
print("Cantidad de registros:", df_raw.count())
print("Cantidad de columnas:", len(df_raw.columns))

df_raw.show(5, truncate=False)

Dataset RAW cargado desde S3 con PySpark.
Cantidad de registros: 783244
Cantidad de columnas: 19
+--------+--------+----+-------+-------+----------+----------+------------+-----------+-----------+---------------+-----------+---------+---------+---------------+-------------+--------------+-------------+-------------+
|Site    |Unit    |Year|Week   |Species|Year class|Open Count|Open Biomass|Open Weight|Feed Weight|Temperature Avg|Density Avg|Live Days|Fish Days|Mortality Count|Harvest Count|Ship Out Count|Ship In Count|Close Biomass|
+--------+--------+----+-------+-------+----------+----------+------------+-----------+-----------+---------------+-----------+---------+---------+---------------+-------------+--------------+-------------+-------------+
|Site_001|Unit_001|2022|2022w45|Coho   |2022-C1   |0         |0           |NULL       |35         |10,825         |6,190291888|4        |303040   |114            |NULL         |0             |75838        |247,0301913  |
|Site_001|Unit_001|

In [11]:
# Esta celda muestra el esquema original del dataset leído desde S3.
# En la capa RAW, las columnas normalmente se interpretan como texto antes de aplicar conversiones.

df_raw.printSchema()

root
 |-- Site: string (nullable = true)
 |-- Unit: string (nullable = true)
 |-- Year: string (nullable = true)
 |-- Week: string (nullable = true)
 |-- Species: string (nullable = true)
 |-- Year class: string (nullable = true)
 |-- Open Count: string (nullable = true)
 |-- Open Biomass: string (nullable = true)
 |-- Open Weight: string (nullable = true)
 |-- Feed Weight: string (nullable = true)
 |-- Temperature Avg: string (nullable = true)
 |-- Density Avg: string (nullable = true)
 |-- Live Days: string (nullable = true)
 |-- Fish Days: string (nullable = true)
 |-- Mortality Count: string (nullable = true)
 |-- Harvest Count: string (nullable = true)
 |-- Ship Out Count: string (nullable = true)
 |-- Ship In Count: string (nullable = true)
 |-- Close Biomass: string (nullable = true)



In [12]:
# Esta celda normaliza los nombres de columnas para facilitar la transformación.
# Se convierten a minúsculas, se eliminan tildes, se reemplazan espacios por guion bajo y se evita usar nombres problemáticos como "year".

import re
import unicodedata

def normalize_column_name(name):
    name = name.strip().lower()
    name = unicodedata.normalize("NFKD", name).encode("ascii", "ignore").decode("utf-8")
    name = re.sub(r"[^a-z0-9]+", "_", name)
    name = re.sub(r"_+", "_", name).strip("_")

    if name == "year":
        name = "year_value"

    return name

df_clean = df_raw

for old_name in df_raw.columns:
    new_name = normalize_column_name(old_name)
    df_clean = df_clean.withColumnRenamed(old_name, new_name)

print("Columnas normalizadas:")
print(df_clean.columns)

Columnas normalizadas:
['site', 'unit', 'year_value', 'week', 'species', 'year_class', 'open_count', 'open_biomass', 'open_weight', 'feed_weight', 'temperature_avg', 'density_avg', 'live_days', 'fish_days', 'mortality_count', 'harvest_count', 'ship_out_count', 'ship_in_count', 'close_biomass']


## 4.7 Atributos e indicadores a generar

Aunque el planteamiento original del proyecto considera registros de alimentación automatizada y video submarino, en esta primera implementación se trabajará con la información productiva histórica disponible.

El dataset no contiene eventos detallados de alimentación en tiempo real ni análisis de comportamiento por video. Sin embargo, sí incluye la variable `Feed Weight`, que permite analizar el alimento entregado semanalmente por centro y unidad de cultivo. Por esta razón, los indicadores generados se enfocarán principalmente en evaluar la alimentación en relación con variables productivas como biomasa, mortalidad, temperatura y densidad.

El objetivo de estos indicadores es construir una primera base analítica que permita observar patrones históricos de alimentación y apoyar decisiones operacionales futuras.

| Indicador | Descripción | Relación con el objetivo del proyecto |
|---|---|---|
| `feed_weight_numeric` | Alimento entregado durante la semana, convertido a valor numérico. | Permite usar la alimentación como variable central del análisis. |
| `feed_per_open_biomass` | Relación entre alimento entregado y biomasa inicial. | Ayuda a comparar cuánto alimento se entrega en relación con la biomasa disponible. |
| `feed_per_fish` | Relación entre alimento entregado y cantidad inicial de peces. | Permite comparar la alimentación relativa entre unidades con distinto número de peces. |
| `biomass_variation` | Diferencia entre biomasa de cierre y biomasa inicial. | Permite observar si la biomasa aumentó o disminuyó durante la semana. |
| `biomass_growth_rate` | Variación porcentual de biomasa respecto a la biomasa inicial. | Sirve como aproximación al resultado productivo asociado al periodo alimentado. |
| `mortality_rate` | Porcentaje de mortalidad semanal respecto al conteo inicial de peces. | Permite revisar si semanas con alimentación, temperatura o densidad determinadas presentan mayor mortalidad. |
| `density_level` | Clasificación simple de densidad en baja, media o alta. | Ayuda a contextualizar la alimentación según condiciones productivas de la unidad. |
| `temperature_level` | Clasificación simple de temperatura en baja, media o alta. | Permite analizar la alimentación considerando una variable ambiental relevante. |
| `production_week_id` | Identificador compuesto por año y semana productiva. | Facilita el análisis histórico y la comparación semanal. |

Estos atributos no reemplazan un sistema completo de alimentación automatizada, pero permiten construir una primera capa batch orientada a analizar el uso de alimento dentro del contexto productivo disponible.

En etapas futuras, esta base podría complementarse con eventos de alimentación más detallados, sensores en tiempo real y video submarino, tal como se planteó en el diseño original del proyecto.

In [13]:
# Esta celda convierte columnas numéricas desde texto a valores numéricos.
# Se corrige la coma decimal usada en algunos valores, reemplazándola por punto para que Spark pueda convertir correctamente.

from pyspark.sql import functions as F

def decimal_comma_to_double(column_name):
    value = F.trim(F.col(column_name))

    normalized_value = F.when(
        value.contains(","),
        F.regexp_replace(value, ",", ".")
    ).otherwise(value)

    return normalized_value.cast("double")

integer_columns = [
    "year_value",
    "week",
    "open_count",
    "live_days",
    "fish_days",
    "mortality_count",
    "harvest_count"
]

decimal_columns = [
    "open_biomass",
    "open_weight",
    "feed_weight",
    "temperature_avg",
    "density_avg",
    "close_biomass"
]

df_typed = df_clean

for column_name in integer_columns:
    if column_name in df_typed.columns:
        df_typed = df_typed.withColumn(
            column_name,
            decimal_comma_to_double(column_name).cast("long")
        )

for column_name in decimal_columns:
    if column_name in df_typed.columns:
        df_typed = df_typed.withColumn(
            column_name,
            decimal_comma_to_double(column_name)
        )

print("Conversión de tipos completada.")
df_typed.printSchema()

Conversión de tipos completada.
root
 |-- site: string (nullable = true)
 |-- unit: string (nullable = true)
 |-- year_value: long (nullable = true)
 |-- week: long (nullable = true)
 |-- species: string (nullable = true)
 |-- year_class: string (nullable = true)
 |-- open_count: long (nullable = true)
 |-- open_biomass: double (nullable = true)
 |-- open_weight: double (nullable = true)
 |-- feed_weight: double (nullable = true)
 |-- temperature_avg: double (nullable = true)
 |-- density_avg: double (nullable = true)
 |-- live_days: long (nullable = true)
 |-- fish_days: long (nullable = true)
 |-- mortality_count: long (nullable = true)
 |-- harvest_count: long (nullable = true)
 |-- ship_out_count: string (nullable = true)
 |-- ship_in_count: string (nullable = true)
 |-- close_biomass: double (nullable = true)



In [14]:
# Esta celda limpia campos descriptivos eliminando espacios innecesarios.
# Esto permite agrupar correctamente por sitio, unidad, especie y clase de año.

text_columns = ["site", "unit", "species", "year_class"]

for column_name in text_columns:
    if column_name in df_typed.columns:
        df_typed = df_typed.withColumn(
            column_name,
            F.trim(F.col(column_name))
        )

print("Campos de texto limpiados correctamente.")
df_typed.select([col for col in text_columns if col in df_typed.columns]).show(5, truncate=False)

Campos de texto limpiados correctamente.
+--------+--------+-------+----------+
|site    |unit    |species|year_class|
+--------+--------+-------+----------+
|Site_001|Unit_001|Coho   |2022-C1   |
|Site_001|Unit_001|Coho   |2022-C1   |
|Site_001|Unit_001|Coho   |2022-C1   |
|Site_001|Unit_001|Coho   |2022-C1   |
|Site_001|Unit_001|Coho   |2022-C1   |
+--------+--------+-------+----------+
only showing top 5 rows



In [15]:
# Esta celda revisa valores nulos en columnas clave y registros con denominadores inválidos.
# Estas validaciones son importantes porque algunos indicadores dividen por biomasa o cantidad inicial de peces.

key_columns = [
    "site",
    "unit",
    "year_value",
    "week",
    "open_count",
    "open_biomass",
    "feed_weight",
    "close_biomass"
]

validation_exprs = []

for column_name in key_columns:
    if column_name in df_typed.columns:
        validation_exprs.append(
            F.sum(F.when(F.col(column_name).isNull(), 1).otherwise(0)).alias(f"null_{column_name}")
        )

df_typed.select(validation_exprs).show(truncate=False)

invalid_denominators = df_typed.filter(
    (F.col("open_count").isNull()) |
    (F.col("open_count") <= 0) |
    (F.col("open_biomass").isNull()) |
    (F.col("open_biomass") <= 0)
).count()

print("Registros con denominadores inválidos o revisables:", invalid_denominators)

+---------+---------+---------------+---------+---------------+-----------------+----------------+------------------+
|null_site|null_unit|null_year_value|null_week|null_open_count|null_open_biomass|null_feed_weight|null_close_biomass|
+---------+---------+---------------+---------+---------------+-----------------+----------------+------------------+
|0        |0        |0              |783244   |0              |0                |147214          |0                 |
+---------+---------+---------------+---------+---------------+-----------------+----------------+------------------+

Registros con denominadores inválidos o revisables: 59373


In [16]:
# Esta celda crea una bandera de calidad de datos.
# La bandera permite distinguir registros válidos de registros con claves incompletas o denominadores que requieren revisión.

df_validated = df_typed.withColumn(
    "data_quality_flag",
    F.when(
        F.col("site").isNull() |
        F.col("unit").isNull() |
        F.col("year_value").isNull() |
        F.col("week").isNull(),
        F.lit("invalid_key")
    ).when(
        (F.col("open_count").isNull()) |
        (F.col("open_count") <= 0) |
        (F.col("open_biomass").isNull()) |
        (F.col("open_biomass") <= 0),
        F.lit("review_denominator")
    ).otherwise(
        F.lit("ok")
    )
)

df_validated.groupBy("data_quality_flag").count().show()

+-----------------+------+
|data_quality_flag| count|
+-----------------+------+
|      invalid_key|783244|
+-----------------+------+



In [17]:
# Esta celda genera indicadores productivos con foco en alimentación.
# Los indicadores relacionan alimento entregado con biomasa, cantidad inicial de peces,
# mortalidad, temperatura promedio y densidad productiva.

df_features = df_validated.withColumn(
    "production_week_id",
    F.concat_ws(
        "-",
        F.col("year_value").cast("string"),
        F.lpad(F.col("week").cast("string"), 2, "0")
    )
).withColumn(
    "feed_weight_numeric",
    F.col("feed_weight")
).withColumn(
    "feed_per_open_biomass",
    F.when(
        F.col("open_biomass") > 0,
        F.col("feed_weight") / F.col("open_biomass")
    )
).withColumn(
    "feed_per_fish",
    F.when(
        F.col("open_count") > 0,
        F.col("feed_weight") / F.col("open_count")
    )
).withColumn(
    "biomass_variation",
    F.col("close_biomass") - F.col("open_biomass")
).withColumn(
    "biomass_growth_rate",
    F.when(
        F.col("open_biomass") > 0,
        (F.col("close_biomass") - F.col("open_biomass")) / F.col("open_biomass")
    )
).withColumn(
    "mortality_rate",
    F.when(
        F.col("open_count") > 0,
        F.col("mortality_count") / F.col("open_count")
    )
).withColumn(
    "density_level",
    F.when(F.col("density_avg").isNull(), F.lit("sin_dato"))
     .when(F.col("density_avg") < 10, F.lit("baja"))
     .when(F.col("density_avg") < 20, F.lit("media"))
     .otherwise(F.lit("alta"))
).withColumn(
    "temperature_level",
    F.when(F.col("temperature_avg").isNull(), F.lit("sin_dato"))
     .when(F.col("temperature_avg") < 10, F.lit("baja"))
     .when(F.col("temperature_avg") <= 14, F.lit("media"))
     .otherwise(F.lit("alta"))
)

print("Indicadores generados correctamente.")

df_features.select(
    "site",
    "unit",
    "production_week_id",
    "feed_weight_numeric",
    "feed_per_open_biomass",
    "feed_per_fish",
    "biomass_variation",
    "biomass_growth_rate",
    "mortality_rate",
    "density_level",
    "temperature_level",
    "data_quality_flag"
).show(10, truncate=False)

Indicadores generados correctamente.
+--------+--------+------------------+-------------------+---------------------+---------------------+------------------+-------------------+---------------------+-------------+-----------------+-----------------+
|site    |unit    |production_week_id|feed_weight_numeric|feed_per_open_biomass|feed_per_fish        |biomass_variation |biomass_growth_rate|mortality_rate       |density_level|temperature_level|data_quality_flag|
+--------+--------+------------------+-------------------+---------------------+---------------------+------------------+-------------------+---------------------+-------------+-----------------+-----------------+
|Site_001|Unit_001|2022              |35.0               |NULL                 |NULL                 |247.0301913       |NULL               |NULL                 |baja         |media            |invalid_key      |
|Site_001|Unit_001|2022              |85.0               |0.3440874961586122   |0.0011224974908879617|63.33

In [18]:
# Esta celda selecciona las columnas finales que formarán parte del dataset procesado.
# Se conservan variables productivas relevantes y se agregan los indicadores derivados.

final_columns = [
    "site",
    "unit",
    "year_value",
    "week",
    "production_week_id",
    "species",
    "year_class",
    "open_count",
    "open_biomass",
    "open_weight",
    "feed_weight_numeric",
    "temperature_avg",
    "temperature_level",
    "density_avg",
    "density_level",
    "live_days",
    "fish_days",
    "mortality_count",
    "mortality_rate",
    "harvest_count",
    "ship_out_count",
    "ship_in_count",
    "close_biomass",
    "biomass_variation",
    "biomass_growth_rate",
    "feed_per_open_biomass",
    "feed_per_fish",
    "data_quality_flag"
]

available_final_columns = [col for col in final_columns if col in df_features.columns]

df_processed = df_features.select(available_final_columns)

print("Dataset procesado final:")
print("Registros:", df_processed.count())
print("Columnas:", len(df_processed.columns))

df_processed.show(10, truncate=False)

Dataset procesado final:
Registros: 783244
Columnas: 28
+--------+--------+----------+----+------------------+-------+----------+----------+------------+-----------+-------------------+---------------+-----------------+-----------+-------------+---------+---------+---------------+---------------------+-------------+--------------+-------------+-------------+------------------+-------------------+---------------------+---------------------+-----------------+
|site    |unit    |year_value|week|production_week_id|species|year_class|open_count|open_biomass|open_weight|feed_weight_numeric|temperature_avg|temperature_level|density_avg|density_level|live_days|fish_days|mortality_count|mortality_rate       |harvest_count|ship_out_count|ship_in_count|close_biomass|biomass_variation |biomass_growth_rate|feed_per_open_biomass|feed_per_fish        |data_quality_flag|
+--------+--------+----------+----+------------------+-------+----------+----------+------------+-----------+-------------------+---

In [19]:
# Esta celda muestra el esquema final del dataset procesado.
# Permite comprobar que las columnas numéricas quedaron con tipos adecuados y que los indicadores fueron agregados.

df_processed.printSchema()

root
 |-- site: string (nullable = true)
 |-- unit: string (nullable = true)
 |-- year_value: long (nullable = true)
 |-- week: long (nullable = true)
 |-- production_week_id: string (nullable = false)
 |-- species: string (nullable = true)
 |-- year_class: string (nullable = true)
 |-- open_count: long (nullable = true)
 |-- open_biomass: double (nullable = true)
 |-- open_weight: double (nullable = true)
 |-- feed_weight_numeric: double (nullable = true)
 |-- temperature_avg: double (nullable = true)
 |-- temperature_level: string (nullable = false)
 |-- density_avg: double (nullable = true)
 |-- density_level: string (nullable = false)
 |-- live_days: long (nullable = true)
 |-- fish_days: long (nullable = true)
 |-- mortality_count: long (nullable = true)
 |-- mortality_rate: double (nullable = true)
 |-- harvest_count: long (nullable = true)
 |-- ship_out_count: string (nullable = true)
 |-- ship_in_count: string (nullable = true)
 |-- close_biomass: double (nullable = true)
 |-- 

In [20]:
# Esta celda genera un resumen estadístico de los principales indicadores.
# Permite comprobar que las métricas fueron calculadas correctamente y tienen valores analizables.

indicator_columns = [
    "feed_weight_numeric",
    "feed_per_open_biomass",
    "feed_per_fish",
    "biomass_variation",
    "biomass_growth_rate",
    "mortality_rate",
    "temperature_avg",
    "density_avg"
]

available_indicator_columns = [col for col in indicator_columns if col in df_processed.columns]

df_processed.select(available_indicator_columns).describe().show(truncate=False)

+-------+-------------------+---------------------+-------------------+------------------+---------------------+--------------------+------------------+------------------+
|summary|feed_weight_numeric|feed_per_open_biomass|feed_per_fish      |biomass_variation |biomass_growth_rate  |mortality_rate      |temperature_avg   |density_avg       |
+-------+-------------------+---------------------+-------------------+------------------+---------------------+--------------------+------------------+------------------+
|count  |636030             |600287               |600283             |783244            |723993               |544471              |773493            |783244            |
|mean   |1770.1309690042926 |4.066245510784829E8  |0.04374966227334596|40.9966099702275  |1.0040760506651096E10|0.020398710565967624|11.191418188292372|23.066767511595287|
|stddev |3402.7553246456014 |2.3612080960891492E11|2.628626314649729  |16282.838080362435|6.366737303213283E12 |0.6423246150788474  |3.25562

In [21]:
# Esta celda resume la cantidad de registros por bandera de calidad.
# Sirve como evidencia de validación de datos dentro de la etapa de transformación.

df_processed.groupBy("data_quality_flag").count().show()

+-----------------+------+
|data_quality_flag| count|
+-----------------+------+
|      invalid_key|783244|
+-----------------+------+



## 4.8 Resultado esperado de la etapa

Al finalizar esta etapa se obtiene un dataset procesado con:

- columnas normalizadas;
- tipos de datos corregidos;
- valores numéricos convertidos correctamente;
- registros validados;
- indicadores productivos generados;
- métricas orientadas al análisis de alimentación;
- archivo procesado listo para ser almacenado en Amazon S3.

El resultado será guardado en la zona `processed/` del bucket S3, separando claramente los datos originales de los datos transformados.

In [22]:
# Esta celda guarda el dataset procesado directamente en Amazon S3 en formato Parquet.
# Parquet es adecuado para la capa procesada porque conserva tipos de datos y mejora el rendimiento de consulta.

df_processed.write \
    .mode("overwrite") \
    .parquet(S3_PROCESSED_PATH)

print("Dataset procesado guardado correctamente en S3.")
print("Ruta S3:", S3_PROCESSED_PATH)

Dataset procesado guardado correctamente en S3.
Ruta S3: s3a://bigdata-salmonicultura-fabian/processed/productive_kpi/


In [23]:
# Esta celda comprueba que Spark escribió correctamente los archivos Parquet en la zona processed de Amazon S3.
# La validación se realiza con boto3 listando los objetos creados en el prefijo processed.

response_processed = s3.list_objects_v2(
    Bucket=S3_BUCKET,
    Prefix=S3_PROCESSED_PREFIX
)

print("Archivos encontrados en la zona processed:")

if "Contents" in response_processed:
    for obj in response_processed["Contents"]:
        print("-", obj["Key"], "| Tamaño:", obj["Size"], "bytes")
else:
    print("No se encontraron archivos en la zona processed.")

Archivos encontrados en la zona processed:
- processed/productive_kpi/_SUCCESS | Tamaño: 0 bytes
- processed/productive_kpi/part-00000-c56b4d23-f292-4f33-9196-7ead4247c85e-c000.snappy.parquet | Tamaño: 4137735 bytes
- processed/productive_kpi/part-00001-c56b4d23-f292-4f33-9196-7ead4247c85e-c000.snappy.parquet | Tamaño: 4156860 bytes
- processed/productive_kpi/part-00002-c56b4d23-f292-4f33-9196-7ead4247c85e-c000.snappy.parquet | Tamaño: 2742962 bytes
- processed/productive_kpi/part-00003-c56b4d23-f292-4f33-9196-7ead4247c85e-c000.snappy.parquet | Tamaño: 3911160 bytes
- processed/productive_kpi/part-00004-c56b4d23-f292-4f33-9196-7ead4247c85e-c000.snappy.parquet | Tamaño: 4003647 bytes
- processed/productive_kpi/part-00005-c56b4d23-f292-4f33-9196-7ead4247c85e-c000.snappy.parquet | Tamaño: 3935212 bytes
- processed/productive_kpi/part-00006-c56b4d23-f292-4f33-9196-7ead4247c85e-c000.snappy.parquet | Tamaño: 3946851 bytes
- processed/productive_kpi/part-00007-c56b4d23-f292-4f33-9196-7ead4247

### Resultado de la transformación

La etapa de preparación y transformación fue completada correctamente.

PySpark leyó directamente el dataset RAW desde Amazon S3, aplicó limpieza, conversión de tipos, validaciones y generación de indicadores productivos orientados al análisis de alimentación.

El resultado procesado fue guardado directamente en Amazon S3 en formato Parquet, dentro de la zona `processed/`.

Con esto se completa la etapa equivalente a Dataprep en la pauta original, adaptada a una arquitectura AWS y utilizando PySpark como motor de transformación batch.

![Bucket S3 transformado](../screenshots/01_s3_bucket/05_bucket_procesados.png)